# Databricks notebook source

# COMMAND ----------
# MAGIC %md
# MAGIC # Agent 2 — Reconciliation Agent
# MAGIC
# MAGIC Compares row counts and financial aggregates across
# MAGIC Raw, Silver, and Gold layers for each registered source table.
# MAGIC Reads Agent 1 context to distinguish explained discrepancies
# MAGIC from genuine cross-layer drift.
# MAGIC
# MAGIC Reads from:
# MAGIC   uc2_layer_registry_b
# MAGIC   uc1_raw_store_transactions
# MAGIC   uc1_raw_inventory_movements
# MAGIC   uc2_silver_store_transactions
# MAGIC   uc2_silver_inventory_movements
# MAGIC   uc2_gold_daily_store_metrics
# MAGIC   uc2_gold_daily_inventory_summary
# MAGIC   uc1_validation_run_b             (Agent 1 context)
# MAGIC
# MAGIC Writes to:
# MAGIC   uc2_reconciliation_run_b
# MAGIC   uc2_reconciliation_results_b

## Parameters

In [0]:
dbutils.widgets.text(
    "business_date",
    "",
    "Business Date (YYYY-MM-DD, leave blank for yesterday)"
)

dbutils.widgets.text(
    "source_table_filter",
    "",
    "Source Table Filter (leave blank for all registered tables)"
)

from datetime import date, datetime, timedelta

BUSINESS_DATE_STR    = dbutils.widgets.get("business_date").strip()
SOURCE_TABLE_FILTER  = dbutils.widgets.get("source_table_filter").strip()

if not BUSINESS_DATE_STR:
    BUSINESS_DATE = date.today() - timedelta(days=1)
    date_source   = "default (yesterday)"
else:
    try:
        BUSINESS_DATE = date.fromisoformat(BUSINESS_DATE_STR)
        date_source   = "provided parameter"
    except ValueError:
        print(
            f"WARNING: Invalid date format '{BUSINESS_DATE_STR}'. "
            f"Defaulting to yesterday."
        )
        BUSINESS_DATE = date.today() - timedelta(days=1)
        date_source   = "fallback (invalid format)"

RUN_START_TIME = datetime.now()

print("=" * 60)
print("AGENT 2 — RECONCILIATION AGENT")
print("=" * 60)
print(f"Business date   : {BUSINESS_DATE}  ({date_source})")
print(f"Table filter    : {SOURCE_TABLE_FILTER or 'All registered tables'}")
print(f"Started at      : {RUN_START_TIME}")
print("=" * 60)

In [0]:
import uuid
import json
from datetime import date, datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DateType, TimestampType,
    DoubleType, IntegerType, BooleanType, LongType
)
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ── Catalog and Schema ─────────────────────────────────────────────────────

dbutils.widgets.text("catalog", "learning_dev", "Target Catalog")
dbutils.widgets.text("schema", "s819534", "Target Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")

# ── Agent 2 Metadata Tables ────────────────────────────────────────────────

LAYER_REGISTRY        = f"{CATALOG}.{SCHEMA}.uc2_layer_registry_b"
RECONCILIATION_RUN    = f"{CATALOG}.{SCHEMA}.uc2_reconciliation_run_b"
RECONCILIATION_RESULTS= f"{CATALOG}.{SCHEMA}.uc2_reconciliation_results_b"

# ── Agent 1 Output Tables (for context) ───────────────────────────────────

AGENT1_VALIDATION_RUN = f"{CATALOG}.{SCHEMA}.uc1_validation_run_b"

# ── Severity Levels ────────────────────────────────────────────────────────

STATUS_PASS     = "PASS"
STATUS_WARNING  = "WARNING"
STATUS_CRITICAL = "CRITICAL"
STATUS_ERROR    = "ERROR"

SEVERITY_RANK = {
    STATUS_PASS    : 0,
    STATUS_WARNING : 1,
    STATUS_CRITICAL: 2,
    STATUS_ERROR   : 3
}

# ── AI Confidence Threshold ────────────────────────────────────────────────

AI_MIN_FAILED_CHECKS  = 2
AI_MIN_CRITICAL_COUNT = 1

# ── Check Type Enumerations ────────────────────────────────────────────────

CHECK_ROW_COUNT_RAW_SILVER  = "RAW_TO_SILVER_ROW_COUNT"
CHECK_ROW_COUNT_SILVER_GOLD = "SILVER_TO_GOLD_ROW_COUNT"
CHECK_AGGREGATE_RAW_SILVER  = "RAW_TO_SILVER_AGGREGATE"
CHECK_AGGREGATE_SILVER_GOLD = "SILVER_TO_GOLD_AGGREGATE"
CHECK_LAYER_FRESHNESS       = "LAYER_FRESHNESS"

# ── Output Table Schemas ───────────────────────────────────────────────────

RUN_SCHEMA = StructType([
    StructField("reconciliation_run_id",           StringType(),    False),
    StructField("source_fully_qualified_name",      StringType(),    False),
    StructField("business_date",                   DateType(),      False),
    StructField("triggered_by",                    StringType(),    True),
    StructField("agent1_validation_run_id",         StringType(),    True),
    StructField("reconciliation_start_timestamp",   TimestampType(), False),
    StructField("reconciliation_end_timestamp",     TimestampType(), True),
    StructField("reconciliation_duration_seconds",  DoubleType(),    True),
    StructField("overall_status",                  StringType(),    False),
    StructField("total_checks_run",                IntegerType(),   False),
    StructField("total_checks_passed",             IntegerType(),   False),
    StructField("total_checks_warned",             IntegerType(),   False),
    StructField("total_checks_critical",           IntegerType(),   False),
    StructField("ai_enrichment_status",            StringType(),    False),
    StructField("ai_enrichment_timestamp",         TimestampType(), True),
    StructField("alert_sent",                      BooleanType(),   True),
    StructField("alert_sent_timestamp",            TimestampType(), True),
    StructField("alert_channel",                   StringType(),    True),
    StructField("created_timestamp",               TimestampType(), False),
    StructField("created_by",                      StringType(),    False),
])

RESULTS_SCHEMA = StructType([
    StructField("result_id",                       StringType(),    False),
    StructField("reconciliation_run_id",           StringType(),    False),
    StructField("source_fully_qualified_name",      StringType(),    False),
    StructField("business_date",                   DateType(),      False),
    StructField("from_layer",                      StringType(),    False),
    StructField("to_layer",                        StringType(),    False),
    StructField("check_type",                      StringType(),    False),
    StructField("metric_name",                     StringType(),    True),
    StructField("check_status",                    StringType(),    False),
    StructField("source_value",                    DoubleType(),    True),
    StructField("target_value",                    DoubleType(),    True),
    StructField("expected_value",                  DoubleType(),    True),
    StructField("deviation_percentage",            DoubleType(),    True),
    StructField("threshold_warning",               DoubleType(),    True),
    StructField("threshold_critical",              DoubleType(),    True),
    StructField("check_detail",                    StringType(),    True),
    StructField("agent1_overall_status",           StringType(),    True),
    StructField("agent1_failure_category",         StringType(),    True),
    StructField("ai_summary",                      StringType(),    True),
    StructField("ai_recommended_action",           StringType(),    True),
    StructField("ai_failure_category",             StringType(),    True),
    StructField("ai_confidence_score",             DoubleType(),    True),
    StructField("ai_model_version",                StringType(),    True),
    StructField("ai_invocation_id",                StringType(),    True),
    StructField("created_timestamp",               TimestampType(), False),
    StructField("ai_enriched_timestamp",           TimestampType(), True),
])

print("Configuration loaded.")

## Step 1: Load Configuration from Layer Registry

In [0]:
def load_registered_source_tables(source_filter: str = "") -> list:
    """
    Loads distinct source tables that have layers registered.
    Each source table has RAW, SILVER, and GOLD layers.
    """
    df = (
        spark.table(LAYER_REGISTRY)
             .filter(F.col("is_active") == True)
             .select("source_fully_qualified_name")
             .distinct()
    )

    if source_filter:
        df = df.filter(
            F.col("source_fully_qualified_name") == source_filter
        )

    tables = [row["source_fully_qualified_name"] for row in df.collect()]

    print(f"Registered source tables: {len(tables)}")
    for t in tables:
        print(f"  → {t}")

    return tables


def load_layers_for_source(source_fqn: str) -> dict:
    """
    Loads all layer records for a given source table.
    Returns a dict keyed by layer_name: RAW, SILVER, GOLD.
    """
    rows = (
        spark.table(LAYER_REGISTRY)
             .filter(
                 (F.col("source_fully_qualified_name") == source_fqn) &
                 (F.col("is_active") == True)
             )
             .orderBy("layer_name")
             .collect()
    )

    layers = {}
    for row in rows:
        layers[row["layer_name"]] = row.asDict()

    print(f"  Layers loaded: {list(layers.keys())}")
    return layers


def load_agent1_context(
    source_fqn    : str,
    business_date : date
) -> dict:
    """
    Reads Agent 1 validation run results for this source table
    and business date to provide context for reconciliation.

    If Agent 1 has not run for this date or table,
    returns an empty context dict.
    """
    try:
        rows = (
            spark.table(AGENT1_VALIDATION_RUN)
                 .filter(
                     (F.col("fully_qualified_name") == source_fqn) &
                     (F.col("business_date") == F.lit(str(business_date)))
                 )
                 .orderBy(F.col("created_timestamp").desc())
                 .limit(1)
                 .collect()
        )

        if not rows:
            return {
                "available"             : False,
                "validation_run_id"     : None,
                "overall_status"        : None,
                "ai_failure_category"   : None,
                "total_checks_critical" : 0,
                "total_checks_warned"   : 0
            }

        row = rows[0].asDict()
        return {
            "available"             : True,
            "validation_run_id"     : row.get("validation_run_id"),
            "overall_status"        : row.get("overall_status"),
            "ai_failure_category"   : row.get("ai_failure_category"),
            "total_checks_critical" : row.get("total_checks_critical", 0),
            "total_checks_warned"   : row.get("total_checks_warned", 0)
        }

    except Exception as e:
        logger.warning(f"Could not load Agent 1 context: {str(e)}")
        return {
            "available"             : False,
            "validation_run_id"     : None,
            "overall_status"        : None,
            "ai_failure_category"   : None,
            "total_checks_critical" : 0,
            "total_checks_warned"   : 0
        }

def load_check_thresholds(source_fqn: str, check_type: str) -> dict:
    """
    Loads warning and critical thresholds for a given
    source table and check type from uc2_check_config_b.

    Falls back to hardcoded defaults if not configured.
    """
    CHECK_CONFIG = f"{CATALOG}.{SCHEMA}.uc2_check_config_b"

    DEFAULT_THRESHOLDS = {
        "ROW_COUNT" : {"warning": 5.0,  "critical": 15.0, "unit": "PERCENTAGE",
                       "sg_warning": 0.0,  "sg_critical": 5.0},
        "AGGREGATE" : {"warning": 2.0,  "critical": 10.0, "unit": "PERCENTAGE",
                       "sg_warning": 0.01, "sg_critical": 0.5},
        "FRESHNESS" : {"warning": 90.0, "critical": 120.0,"unit": "MINUTES",
                       "sg_warning": 90.0, "sg_critical": 120.0},
    }

    try:
        rows = (
            spark.table(CHECK_CONFIG)
                 .filter(
                     (F.col("source_fully_qualified_name") == source_fqn) &
                     (F.col("check_type") == check_type) &
                     (F.col("is_active") == True)
                 )
                 .limit(1)
                 .collect()
        )

        if not rows:
            logger.warning(
                f"No check config found for {check_type} on {source_fqn}. "
                f"Using defaults."
            )
            return DEFAULT_THRESHOLDS.get(check_type, {
                "warning": 5.0, "critical": 15.0,
                "unit": "PERCENTAGE",
                "sg_warning": 0.01, "sg_critical": 0.5
            })

        row = rows[0].asDict()

        # Parse Silver→Gold specific thresholds from custom_parameters
        sg_warn = 0.01
        sg_crit = 0.5
        try:
            params  = json.loads(row.get("custom_parameters", "{}"))
            sg_warn = float(params.get("silver_to_gold_warning", 0.01))
            sg_crit = float(params.get("silver_to_gold_critical", 0.5))
        except (json.JSONDecodeError, TypeError, ValueError):
            pass

        return {
            "warning"    : float(row["warning_threshold"]),
            "critical"   : float(row["critical_threshold"]),
            "unit"       : row["threshold_unit"],
            "sg_warning" : sg_warn,
            "sg_critical": sg_crit
        }

    except Exception as e:
        logger.warning(
            f"Could not load check config for {check_type}: {str(e)}. "
            f"Using defaults."
        )
        return DEFAULT_THRESHOLDS.get(check_type, {
            "warning": 5.0, "critical": 15.0,
            "unit": "PERCENTAGE",
            "sg_warning": 0.01, "sg_critical": 0.5
        })

print("Configuration loaders ready.")

## Step 2: Row Count and Aggregate Helper Functions

In [0]:
def get_row_count(
    table_fqn         : str,
    date_column       : str,
    business_date     : date
) -> int:
    """
    Returns the row count for a given table and business date.
    Returns 0 if the table has no data for that date.
    Returns -1 if the table does not exist or query fails.
    """
    try:
        df = spark.table(table_fqn)

        if date_column and date_column in df.columns:
            count = df.filter(
                F.col(date_column) == F.lit(str(business_date))
            ).count()
        else:
            count = df.count()

        return count

    except Exception as e:
        logger.error(f"Failed to count rows in {table_fqn}: {str(e)}")
        return -1


def get_aggregate(
    table_fqn     : str,
    date_column   : str,
    business_date : date,
    metric_column : str
) -> float:
    """
    Returns the SUM of a metric column for a given table and date.
    Returns None if the column does not exist or query fails.
    """
    try:
        df = spark.table(table_fqn)

        if metric_column not in df.columns:
            logger.warning(
                f"Column {metric_column} not found in {table_fqn}"
            )
            return None

        if date_column and date_column in df.columns:
            df = df.filter(
                F.col(date_column) == F.lit(str(business_date))
            )

        result = df.agg(
            F.sum(metric_column).alias("total")
        ).collect()[0]["total"]

        return float(result) if result is not None else 0.0

    except Exception as e:
        logger.error(
            f"Failed to aggregate {metric_column} in {table_fqn}: {str(e)}"
        )
        return None


def get_load_timestamp(
    table_fqn   : str,
    date_column : str,
    business_date: date,
    ts_column   : str
) -> datetime:
    """
    Returns the MAX of a timestamp column for a given table and date.
    Used for freshness checks.
    """
    try:
        df = spark.table(table_fqn)

        if ts_column not in df.columns:
            return None

        if date_column and date_column in df.columns:
            df = df.filter(
                F.col(date_column) == F.lit(str(business_date))
            )

        result = df.agg(
            F.max(ts_column).alias("max_ts")
        ).collect()[0]["max_ts"]

        return result

    except Exception as e:
        logger.error(f"Failed to get timestamp from {table_fqn}: {str(e)}")
        return None


def build_result(
    reconciliation_run_id  : str,
    source_fqn             : str,
    business_date          : date,
    from_layer             : str,
    to_layer               : str,
    check_type             : str,
    metric_name            : str,
    check_status           : str,
    source_value           : float,
    target_value           : float,
    expected_value         : float,
    deviation_pct          : float,
    threshold_warning      : float,
    threshold_critical     : float,
    check_detail           : dict,
    agent1_context         : dict
) -> dict:
    """
    Builds a single reconciliation result record.
    """
    return {
        "result_id"                  : str(uuid.uuid4()),
        "reconciliation_run_id"      : reconciliation_run_id,
        "source_fully_qualified_name": source_fqn,
        "business_date"              : business_date,
        "from_layer"                 : from_layer,
        "to_layer"                   : to_layer,
        "check_type"                 : check_type,
        "metric_name"                : metric_name,
        "check_status"               : check_status,
        "source_value"               : source_value,
        "target_value"               : target_value,
        "expected_value"             : expected_value,
        "deviation_percentage"       : deviation_pct,
        "threshold_warning"          : threshold_warning,
        "threshold_critical"         : threshold_critical,
        "check_detail"               : json.dumps(check_detail),
        "agent1_overall_status"      : agent1_context.get("overall_status"),
        "agent1_failure_category"    : agent1_context.get("ai_failure_category"),
        "ai_summary"                 : None,
        "ai_recommended_action"      : None,
        "ai_failure_category"        : None,
        "ai_confidence_score"        : None,
        "ai_model_version"           : None,
        "ai_invocation_id"           : None,
        "created_timestamp"          : datetime.now(),
        "ai_enriched_timestamp"      : None
    }


print("Row count and aggregate helpers ready.")

## Step 3: Reconciliation Check Implementations

In [0]:
def check_raw_to_silver_row_count(
    source_fqn     : str,
    layers         : dict,
    business_date  : date,
    agent1_context : dict,
    run_id         : str
) -> dict:
    """
    Check: RAW_TO_SILVER_ROW_COUNT

    Compares the raw row count against the silver row count.
    Calculates the actual survival rate and compares against
    the expected survival rate configured in the layer registry.

    Thresholds applied to the deviation from expected survival:
      Warning  if actual survival deviates > 5% from expected
      Critical if actual survival deviates > 15% from expected
    """
    print(f"    [{CHECK_ROW_COUNT_RAW_SILVER}] Running...")

    raw_layer    = layers.get("RAW", {})
    silver_layer = layers.get("SILVER", {})

    if not raw_layer or not silver_layer:
        print(f"    Missing RAW or SILVER layer config. Skipping.")
        return None

    raw_table    = raw_layer.get("layer_fully_qualified_name")
    silver_table = silver_layer.get("layer_fully_qualified_name")
    raw_date_col = raw_layer.get("business_date_column")
    sil_date_col = silver_layer.get("business_date_column")
    expected_survival = float(silver_layer.get("expected_survival_rate", 0.92))

    # warn_threshold = 5.0   # % deviation from expected survival
    # crit_threshold = 15.0

    thresholds   = load_check_thresholds(source_fqn, "ROW_COUNT")
    warn_threshold = thresholds["warning"]
    crit_threshold = thresholds["critical"]

    raw_count    = get_row_count(raw_table, raw_date_col, business_date)
    silver_count = get_row_count(silver_table, sil_date_col, business_date)

    if raw_count <= 0:
        print(f"    No raw data for {business_date}. Skipping.")
        return None

    if silver_count < 0:
        check_status = STATUS_CRITICAL
        actual_survival = 0.0
        deviation = 100.0
        detail = {
            "raw_count"          : raw_count,
            "silver_count"       : "TABLE_ERROR",
            "actual_survival_pct": 0.0,
            "expected_survival_pct": expected_survival * 100,
            "error"              : "Could not query Silver table"
        }
    else:
        actual_survival = silver_count / raw_count
        deviation = (actual_survival - expected_survival) / expected_survival * 100
        # Positive = survival ABOVE expected (acceptable)
        # Negative = survival BELOW expected (problem)

        # Only flag when survival is BELOW expected
        abs_negative_deviation = abs(min(deviation, 0))

        if silver_count == 0:
            check_status = STATUS_CRITICAL
        elif abs_negative_deviation >= crit_threshold:
            check_status = STATUS_CRITICAL
        elif abs_negative_deviation >= warn_threshold:
            check_status = STATUS_WARNING
        else:
            check_status = STATUS_PASS

        detail = {
            "raw_count"              : raw_count,
            "silver_count"           : silver_count,
            "actual_survival_pct"    : round(actual_survival * 100, 2),
            "expected_survival_pct"  : round(expected_survival * 100, 2),
            "deviation_pct"          : round(deviation, 2),
            "direction"              : "ABOVE_EXPECTED" if deviation >= 0 else "BELOW_EXPECTED",
            "raw_table"              : raw_table,
            "silver_table"           : silver_table,
            "agent1_context_available": agent1_context.get("available"),
            "agent1_status"          : agent1_context.get("overall_status")
        }

    print(f"      Raw count     : {raw_count}")
    print(f"      Silver count  : {silver_count}")
    print(f"      Survival rate : {actual_survival*100:.2f}%")
    print(f"      Expected      : {expected_survival*100:.2f}%")
    print(f"      Deviation     : {deviation:.2f}%")
    print(f"      Status        : {check_status}")
    if agent1_context.get("available"):
        print(
            f"      Agent 1       : {agent1_context.get('overall_status')} "
            f"on raw layer"
        )

    return build_result(
        reconciliation_run_id  = run_id,
        source_fqn             = source_fqn,
        business_date          = business_date,
        from_layer             = "RAW",
        to_layer               = "SILVER",
        check_type             = CHECK_ROW_COUNT_RAW_SILVER,
        metric_name            = "row_count_survival_rate",
        check_status           = check_status,
        source_value           = float(raw_count),
        target_value           = float(silver_count),
        expected_value         = round(raw_count * expected_survival, 0),
        deviation_pct          = round(deviation, 4),
        threshold_warning      = warn_threshold,
        threshold_critical     = crit_threshold,
        check_detail           = detail,
        agent1_context         = agent1_context
    )


def check_silver_to_gold_row_count(
    source_fqn     : str,
    layers         : dict,
    business_date  : date,
    agent1_context : dict,
    run_id         : str
) -> dict:
    """
    Check: SILVER_TO_GOLD_ROW_COUNT

    For transactions: Gold should have one row per store per date.
    Expected Gold rows = COUNT(DISTINCT store_id) in Silver.

    For inventory: Gold should have one row per warehouse
    per event_type per date.
    Expected Gold rows = COUNT(DISTINCT warehouse_id, event_type) in Silver.

    Thresholds:
      Warning  if any deviation > 0
      Critical if deviation > 5%
    """
    print(f"    [{CHECK_ROW_COUNT_SILVER_GOLD}] Running...")

    silver_layer = layers.get("SILVER", {})
    gold_layer   = layers.get("GOLD", {})

    if not silver_layer or not gold_layer:
        print(f"    Missing SILVER or GOLD layer config. Skipping.")
        return None

    silver_table = silver_layer.get("layer_fully_qualified_name")
    gold_table   = gold_layer.get("layer_fully_qualified_name")
    sil_date_col = silver_layer.get("business_date_column")
    gld_date_col = gold_layer.get("business_date_column")

    # warn_threshold = 0.0
    # crit_threshold = 5.0

    thresholds   = load_check_thresholds(source_fqn, "ROW_COUNT")
    warn_threshold = thresholds["warning"]
    crit_threshold = thresholds["critical"]

    gold_count = get_row_count(gold_table, gld_date_col, business_date)

    # Determine expected Gold rows based on source table type
    try:
        silver_df = spark.table(silver_table).filter(
            F.col(sil_date_col) == F.lit(str(business_date))
        )

        if "transaction_id" in silver_df.columns:
            # Transactions: one row per store per date
            expected_gold = silver_df.select(
                "store_id"
            ).distinct().count()
            grouping_desc = "distinct stores in Silver"
        elif "event_id" in silver_df.columns:
            # Inventory: one row per warehouse per event_type per date
            expected_gold = silver_df.select(
                "warehouse_id", "event_type"
            ).distinct().count()
            grouping_desc = "distinct warehouse+event_type combos in Silver"
        else:
            expected_gold = None
            grouping_desc = "unknown"

    except Exception as e:
        logger.error(f"Failed to compute expected Gold rows: {str(e)}")
        expected_gold = None
        grouping_desc = "error"

    if gold_count < 0:
        check_status = STATUS_CRITICAL
        deviation    = 100.0
        detail = {
            "gold_count"    : "TABLE_ERROR",
            "expected_gold" : expected_gold,
            "grouping"      : grouping_desc,
            "error"         : "Could not query Gold table"
        }
    elif gold_count == 0:
        check_status = STATUS_CRITICAL
        deviation    = 100.0
        detail = {
            "gold_count"  : 0,
            "expected_gold": expected_gold,
            "grouping"    : grouping_desc,
            "note"        : "Gold table has zero rows for this date"
        }
    elif expected_gold is not None:
        deviation = abs(gold_count - expected_gold) / max(expected_gold, 1) * 100

        if deviation > crit_threshold:
            check_status = STATUS_CRITICAL
        elif deviation > warn_threshold:
            check_status = STATUS_WARNING
        else:
            check_status = STATUS_PASS

        detail = {
            "gold_count"    : gold_count,
            "expected_gold" : expected_gold,
            "grouping"      : grouping_desc,
            "deviation_pct" : round(deviation, 2),
            "gold_table"    : gold_table
        }
    else:
        check_status = STATUS_WARNING
        deviation    = 0.0
        detail = {
            "gold_count"    : gold_count,
            "expected_gold" : None,
            "note"          : "Could not compute expected Gold rows"
        }

    print(f"      Gold count    : {gold_count}")
    print(f"      Expected      : {expected_gold} ({grouping_desc})")
    print(f"      Status        : {check_status}")

    return build_result(
        reconciliation_run_id  = run_id,
        source_fqn             = source_fqn,
        business_date          = business_date,
        from_layer             = "SILVER",
        to_layer               = "GOLD",
        check_type             = CHECK_ROW_COUNT_SILVER_GOLD,
        metric_name            = "gold_row_count",
        check_status           = check_status,
        source_value           = float(expected_gold or 0),
        target_value           = float(gold_count),
        expected_value         = float(expected_gold or 0),
        deviation_pct          = round(deviation, 4),
        threshold_warning      = warn_threshold,
        threshold_critical     = crit_threshold,
        check_detail           = detail,
        agent1_context         = agent1_context
    )


def check_raw_to_silver_aggregate(
    source_fqn     : str,
    layers         : dict,
    business_date  : date,
    agent1_context : dict,
    run_id         : str
) -> list:
    """
    Check: RAW_TO_SILVER_AGGREGATE

    Compares the SUM of configured financial metrics
    between Raw and Silver layers.

    For transactions: SUM(net_amount)
    For inventory: SUM(quantity_change), SUM(total_cost_impact)

    Tolerance: Silver aggregate should be within 2% of Raw aggregate.
    VOIDs have zero net_amount so their exclusion in Silver
    should not change the sum significantly.

    Thresholds:
      Warning  if deviation > 2%
      Critical if deviation > 10%
    """
    print(f"    [{CHECK_AGGREGATE_RAW_SILVER}] Running...")

    raw_layer    = layers.get("RAW", {})
    silver_layer = layers.get("SILVER", {})

    if not raw_layer or not silver_layer:
        return []

    raw_table    = raw_layer.get("layer_fully_qualified_name")
    silver_table = silver_layer.get("layer_fully_qualified_name")
    raw_date_col = raw_layer.get("business_date_column")
    sil_date_col = silver_layer.get("business_date_column")

    # # Determine appropriate thresholds based on source table type
    # # Inventory tables have higher expected variance due to
    # # DELETE record exclusion and malformed record filtering
    # if "inventory" in source_fqn.lower():
    #     warn_threshold = 15.0   # wider for inventory
    #     crit_threshold = 25.0
    # else:
    #     warn_threshold = 2.0    # tight for transactions
    #     crit_threshold = 10.0

    thresholds   = load_check_thresholds(source_fqn, "AGGREGATE")
    warn_threshold = thresholds["warning"]
    crit_threshold = thresholds["critical"]

    # Parse metrics from layer registry
    try:
        raw_metrics = json.loads(
            raw_layer.get("reconciliation_metrics", "[]")
        )
    except Exception:
        raw_metrics = []

    try:
        sil_metrics = json.loads(
            silver_layer.get("reconciliation_metrics", "[]")
        )
    except Exception:
        sil_metrics = []

    # Use Raw metrics as the reference list
    metrics_to_check = raw_metrics if raw_metrics else sil_metrics

    results = []

    for metric in metrics_to_check:
        print(f"      Checking metric: {metric}")

        raw_total    = get_aggregate(raw_table, raw_date_col, business_date, metric)
        silver_total = get_aggregate(silver_table, sil_date_col, business_date, metric)

        if raw_total is None or silver_total is None:
            print(f"      Could not get aggregate for {metric}. Skipping.")
            continue

        if raw_total == 0:
            deviation = 0.0
        else:
            deviation = abs(raw_total - silver_total) / abs(raw_total) * 100

        if deviation >= crit_threshold:
            check_status = STATUS_CRITICAL
        elif deviation >= warn_threshold:
            check_status = STATUS_WARNING
        else:
            check_status = STATUS_PASS

        detail = {
            "metric"          : metric,
            "raw_total"       : round(raw_total, 2),
            "silver_total"    : round(silver_total, 2),
            "difference"      : round(raw_total - silver_total, 2),
            "deviation_pct"   : round(deviation, 4),
            "raw_table"       : raw_table,
            "silver_table"    : silver_table,
            "note"            : (
                "Small differences expected due to VOID exclusion "
                "and null net_amount derivation in Silver"
            )
        }

        print(f"        Raw total    : {raw_total:,.2f}")
        print(f"        Silver total : {silver_total:,.2f}")
        print(f"        Deviation    : {deviation:.4f}%")
        print(f"        Status       : {check_status}")

        results.append(
            build_result(
                reconciliation_run_id  = run_id,
                source_fqn             = source_fqn,
                business_date          = business_date,
                from_layer             = "RAW",
                to_layer               = "SILVER",
                check_type             = CHECK_AGGREGATE_RAW_SILVER,
                metric_name            = metric,
                check_status           = check_status,
                source_value           = round(raw_total, 4),
                target_value           = round(silver_total, 4),
                expected_value         = round(raw_total, 4),
                deviation_pct          = round(deviation, 4),
                threshold_warning      = warn_threshold,
                threshold_critical     = crit_threshold,
                check_detail           = detail,
                agent1_context         = agent1_context
            )
        )

    return results


def check_silver_to_gold_aggregate(
    source_fqn     : str,
    layers         : dict,
    business_date  : date,
    agent1_context : dict,
    run_id         : str
) -> list:
    """
    Check: SILVER_TO_GOLD_AGGREGATE

    Compares the SUM of financial metrics between Silver and Gold.
    Since Gold is a direct aggregation of Silver, these should match
    exactly (within floating point tolerance).

    For transactions:
      Silver SUM(net_amount) = Gold SUM(total_net_amount)

    For inventory:
      Silver SUM(quantity_change) = Gold SUM(net_quantity_change)
      Silver SUM(total_cost_impact) = Gold SUM(total_cost_impact)

    Thresholds:
      Warning  if any deviation exists
      Critical if deviation > 0.5%
    """
    print(f"    [{CHECK_AGGREGATE_SILVER_GOLD}] Running...")

    silver_layer = layers.get("SILVER", {})
    gold_layer   = layers.get("GOLD", {})

    if not silver_layer or not gold_layer:
        return []

    silver_table = silver_layer.get("layer_fully_qualified_name")
    gold_table   = gold_layer.get("layer_fully_qualified_name")
    sil_date_col = silver_layer.get("business_date_column")
    gld_date_col = gold_layer.get("business_date_column")

    # warn_threshold = 0.01
    # crit_threshold = 0.5

    thresholds   = load_check_thresholds(source_fqn, "AGGREGATE")
    warn_threshold = thresholds["warning"]
    crit_threshold = thresholds["critical"]

    # Metric mapping between Silver and Gold column names
    # Silver column → Gold column
    if "store_transactions" in source_fqn:
        metric_map = {
            "net_amount"      : "total_net_amount",
            "gross_amount"    : "total_gross_amount",
            "discount_amount" : "total_discount_amount"
        }
    elif "inventory" in source_fqn:
        metric_map = {
            "quantity_change"   : "net_quantity_change",
            "total_cost_impact" : "total_cost_impact"
        }
    else:
        metric_map = {}

    results = []

    for silver_metric, gold_metric in metric_map.items():
        print(f"      Checking: Silver.{silver_metric} vs Gold.{gold_metric}")

        silver_total = get_aggregate(
            silver_table, sil_date_col, business_date, silver_metric
        )
        gold_total = get_aggregate(
            gold_table, gld_date_col, business_date, gold_metric
        )

        if silver_total is None:
            print(f"      Silver metric {silver_metric} not available. Skipping.")
            continue

        if gold_total is None:
            check_status = STATUS_CRITICAL
            deviation    = 100.0
            detail = {
                "silver_metric"  : silver_metric,
                "gold_metric"    : gold_metric,
                "silver_total"   : round(silver_total, 2) if silver_total else None,
                "gold_total"     : "NOT_AVAILABLE",
                "note"           : "Gold table missing or metric column not found"
            }
        elif silver_total == 0 and gold_total == 0:
            check_status = STATUS_PASS
            deviation    = 0.0
            detail = {
                "silver_metric"  : silver_metric,
                "gold_metric"    : gold_metric,
                "silver_total"   : 0.0,
                "gold_total"     : 0.0,
                "deviation_pct"  : 0.0
            }
        else:
            if silver_total == 0:
                deviation = 100.0 if gold_total != 0 else 0.0
            else:
                deviation = abs(silver_total - gold_total) / abs(silver_total) * 100

            if deviation >= crit_threshold:
                check_status = STATUS_CRITICAL
            elif deviation >= warn_threshold:
                check_status = STATUS_WARNING
            else:
                check_status = STATUS_PASS

            detail = {
                "silver_metric"  : silver_metric,
                "gold_metric"    : gold_metric,
                "silver_total"   : round(silver_total, 2),
                "gold_total"     : round(gold_total, 2),
                "difference"     : round(silver_total - gold_total, 2),
                "deviation_pct"  : round(deviation, 6)
            }

        print(f"        Silver total : {silver_total:,.2f}" if silver_total else "")
        print(f"        Gold total   : {gold_total:,.2f}" if gold_total else "        Gold total   : NOT AVAILABLE")
        print(f"        Deviation    : {deviation:.6f}%")
        print(f"        Status       : {check_status}")

        results.append(
            build_result(
                reconciliation_run_id  = run_id,
                source_fqn             = source_fqn,
                business_date          = business_date,
                from_layer             = "SILVER",
                to_layer               = "GOLD",
                check_type             = CHECK_AGGREGATE_SILVER_GOLD,
                metric_name            = f"{silver_metric}_vs_{gold_metric}",
                check_status           = check_status,
                source_value           = round(silver_total or 0, 4),
                target_value           = round(gold_total or 0, 4),
                expected_value         = round(silver_total or 0, 4),
                deviation_pct          = round(deviation, 6),
                threshold_warning      = warn_threshold,
                threshold_critical     = crit_threshold,
                check_detail           = detail,
                agent1_context         = agent1_context
            )
        )

    return results


def check_layer_freshness(
    source_fqn     : str,
    layers         : dict,
    business_date  : date,
    agent1_context : dict,
    run_id         : str
) -> list:
    """
    Check: LAYER_FRESHNESS

    For synthetic PoC data: checks whether Gold data exists
    for the business date at all, and whether Silver exists.
    Does not compare actual timestamps since synthetic data
    was generated at different wall-clock times.

    For production: would compare actual load timestamps.
    """
    print(f"    [{CHECK_LAYER_FRESHNESS}] Running...")

    results = []

    silver_layer = layers.get("SILVER", {})
    gold_layer   = layers.get("GOLD", {})

    if not silver_layer or not gold_layer:
        return []

    silver_table = silver_layer.get("layer_fully_qualified_name")
    gold_table   = gold_layer.get("layer_fully_qualified_name")
    sil_date_col = silver_layer.get("business_date_column")
    gld_date_col = gold_layer.get("business_date_column")
    expected_latency = float(gold_layer.get("expected_latency_minutes", 60))

    # warn_threshold = expected_latency * 1.5
    # crit_threshold = expected_latency * 2.0

    thresholds   = load_check_thresholds(source_fqn, "FRESHNESS")
    warn_threshold = thresholds["warning"]
    crit_threshold = thresholds["critical"]

    # ── Check Silver exists for this business date ─────────────────────────
    silver_count = get_row_count(silver_table, sil_date_col, business_date)

    if silver_count == 0:
        check_status = STATUS_CRITICAL
        detail = {
            "silver_table"   : silver_table,
            "business_date"  : str(business_date),
            "silver_count"   : 0,
            "note"           : "Silver has no records for this business date."
        }
        print(f"      Silver missing for {business_date} → CRITICAL")
    else:
        check_status = STATUS_PASS
        detail = {
            "silver_table"   : silver_table,
            "silver_count"   : silver_count,
            "note"           : "Silver exists for this business date."
        }
        print(f"      Silver present: {silver_count} records → PASS")

    results.append(
        build_result(
            reconciliation_run_id  = run_id,
            source_fqn             = source_fqn,
            business_date          = business_date,
            from_layer             = "RAW",
            to_layer               = "SILVER",
            check_type             = CHECK_LAYER_FRESHNESS,
            metric_name            = "silver_data_presence",
            check_status           = check_status,
            source_value           = float(silver_count),
            target_value           = float(silver_count),
            expected_value         = float(silver_count),
            deviation_pct          = 0.0 if silver_count > 0 else 100.0,
            threshold_warning      = warn_threshold,
            threshold_critical     = crit_threshold,
            check_detail           = detail,
            agent1_context         = agent1_context
        )
    )

    # ── Check Gold exists for this business date ───────────────────────────
    gold_count = get_row_count(gold_table, gld_date_col, business_date)

    if gold_count == 0:
        check_status = STATUS_CRITICAL
        detail = {
            "gold_table"    : gold_table,
            "business_date" : str(business_date),
            "gold_count"    : 0,
            "note"          : (
                "Gold has no records for this business date. "
                "Gold pipeline may not have run or may have failed."
            )
        }
        print(f"      Gold missing for {business_date} → CRITICAL")
    else:
        check_status = STATUS_PASS
        detail = {
            "gold_table"  : gold_table,
            "gold_count"  : gold_count,
            "note"        : "Gold exists for this business date."
        }
        print(f"      Gold present: {gold_count} records → PASS")

    results.append(
        build_result(
            reconciliation_run_id  = run_id,
            source_fqn             = source_fqn,
            business_date          = business_date,
            from_layer             = "SILVER",
            to_layer               = "GOLD",
            check_type             = CHECK_LAYER_FRESHNESS,
            metric_name            = "gold_data_presence",
            check_status           = check_status,
            source_value           = float(gold_count),
            target_value           = float(gold_count),
            expected_value         = float(gold_count),
            deviation_pct          = 0.0 if gold_count > 0 else 100.0,
            threshold_warning      = warn_threshold,
            threshold_critical     = crit_threshold,
            check_detail           = detail,
            agent1_context         = agent1_context
        )
    )

    return results

print("All check functions loaded.")

## Step 4: Write Results to Output Tables

In [0]:
def write_reconciliation_results(results: list) -> bool:
    if not results:
        return True

    col_order = [f.name for f in RESULTS_SCHEMA.fields]
    rows      = [tuple(r.get(col) for col in col_order) for r in results]
    df        = spark.createDataFrame(rows, schema=RESULTS_SCHEMA)

    df.write.mode("append").partitionBy("business_date").saveAsTable(
        RECONCILIATION_RESULTS
    )
    print(f"  Written {len(results)} result records.")
    return True


def write_reconciliation_run(run_record: dict) -> bool:
    col_order = [f.name for f in RUN_SCHEMA.fields]
    rows      = [tuple(run_record.get(col) for col in col_order)]
    df        = spark.createDataFrame(rows, schema=RUN_SCHEMA)

    df.write.mode("append").partitionBy("business_date").saveAsTable(
        RECONCILIATION_RUN
    )
    print(f"  Written run record.")
    return True


def determine_overall_status(results: list) -> str:
    if not results:
        return STATUS_PASS
    worst = STATUS_PASS
    for r in results:
        if SEVERITY_RANK.get(r["check_status"], 0) > SEVERITY_RANK.get(worst, 0):
            worst = r["check_status"]
    return worst


def evaluate_ai_confidence(results: list) -> dict:
    failed   = [r for r in results if r["check_status"] in (STATUS_WARNING, STATUS_CRITICAL)]
    critical = [r for r in results if r["check_status"] == STATUS_CRITICAL]

    threshold_met = (
        len(failed)   >= AI_MIN_FAILED_CHECKS and
        len(critical) >= AI_MIN_CRITICAL_COUNT
    )

    return {
        "threshold_met"       : threshold_met,
        "failed_check_count"  : len(failed),
        "critical_check_count": len(critical),
        "failed_checks"       : [r["check_type"] for r in failed],
        "skip_reason"         : None if threshold_met else (
            "INSUFFICIENT_FAILURES"
            if len(failed) < AI_MIN_FAILED_CHECKS
            else "NO_CRITICAL_CHECKS"
        )
    }


print("Write functions ready.")

## Step 5: Run Reconciliation for All Registered Tables

In [0]:
source_tables      = load_registered_source_tables(SOURCE_TABLE_FILTER)
all_run_records    = []
all_result_records = []

print(f"\nReconciling {len(source_tables)} table(s) for {BUSINESS_DATE}\n")

for source_fqn in source_tables:

    TABLE_RUN_ID = str(uuid.uuid4())

    print(f"\n{'='*60}")
    print(f"Source table   : {source_fqn}")
    print(f"Reconciliation : {TABLE_RUN_ID}")
    print(f"Business date  : {BUSINESS_DATE}")
    print(f"{'='*60}")

    # ── Load configuration ─────────────────────────────────────────────────
    layers         = load_layers_for_source(source_fqn)
    agent1_context = load_agent1_context(source_fqn, BUSINESS_DATE)

    print(
        f"  Agent 1 context: "
        f"{'Available' if agent1_context['available'] else 'Not available'}"
    )
    if agent1_context["available"]:
        print(
            f"  Agent 1 status : {agent1_context['overall_status']}"
        )

    table_results = []

    # ── Run all checks ─────────────────────────────────────────────────────
    print(f"\n  Running checks...")

    # Check 1: Raw to Silver row count
    result = check_raw_to_silver_row_count(
        source_fqn, layers, BUSINESS_DATE, agent1_context, TABLE_RUN_ID
    )
    if result:
        table_results.append(result)

    # Check 2: Silver to Gold row count
    result = check_silver_to_gold_row_count(
        source_fqn, layers, BUSINESS_DATE, agent1_context, TABLE_RUN_ID
    )
    if result:
        table_results.append(result)

    # Check 3: Raw to Silver aggregate
    agg_results = check_raw_to_silver_aggregate(
        source_fqn, layers, BUSINESS_DATE, agent1_context, TABLE_RUN_ID
    )
    table_results.extend(agg_results)

    # Check 4: Silver to Gold aggregate
    agg_results = check_silver_to_gold_aggregate(
        source_fqn, layers, BUSINESS_DATE, agent1_context, TABLE_RUN_ID
    )
    table_results.extend(agg_results)

    # Check 5: Layer freshness
    fresh_results = check_layer_freshness(
        source_fqn, layers, BUSINESS_DATE, agent1_context, TABLE_RUN_ID
    )
    table_results.extend(fresh_results)

    # ── Evaluate results ───────────────────────────────────────────────────
    overall_status = determine_overall_status(table_results)
    confidence     = evaluate_ai_confidence(table_results)
    ai_status      = "PENDING" if confidence["threshold_met"] else "NOT_REQUIRED"

    checks_passed   = sum(1 for r in table_results if r["check_status"] == STATUS_PASS)
    checks_warned   = sum(1 for r in table_results if r["check_status"] == STATUS_WARNING)
    checks_critical = sum(1 for r in table_results if r["check_status"] == STATUS_CRITICAL)

    run_end_time = datetime.now()
    duration     = (run_end_time - RUN_START_TIME).total_seconds()

    # ── Build run record ───────────────────────────────────────────────────
    run_record = {
        "reconciliation_run_id"           : TABLE_RUN_ID,
        "source_fully_qualified_name"      : source_fqn,
        "business_date"                   : BUSINESS_DATE,
        "triggered_by"                    : "MANUAL",
        "agent1_validation_run_id"         : agent1_context.get("validation_run_id"),
        "reconciliation_start_timestamp"   : RUN_START_TIME,
        "reconciliation_end_timestamp"     : run_end_time,
        "reconciliation_duration_seconds"  : duration,
        "overall_status"                  : overall_status,
        "total_checks_run"                : len(table_results),
        "total_checks_passed"             : checks_passed,
        "total_checks_warned"             : checks_warned,
        "total_checks_critical"           : checks_critical,
        "ai_enrichment_status"            : ai_status,
        "ai_enrichment_timestamp"         : None,
        "alert_sent"                      : None,
        "alert_sent_timestamp"            : None,
        "alert_channel"                   : None,
        "created_timestamp"               : datetime.now(),
        "created_by"                      : "01_reconciliation_agent"
    }

    all_run_records.append(run_record)
    all_result_records.extend(table_results)

    # ── Print summary ──────────────────────────────────────────────────────
    print(f"\n  {'─'*50}")
    print(f"  RESULTS: {source_fqn.split('.')[-1]}")
    print(f"  {'─'*50}")
    print(f"  Overall status    : {overall_status}")
    print(f"  Checks run        : {len(table_results)}")
    print(f"  Passed            : {checks_passed}")
    print(f"  Warning           : {checks_warned}")
    print(f"  Critical          : {checks_critical}")
    print(f"  AI enrichment     : {ai_status}")
    if confidence["threshold_met"]:
        print(f"  AI triggered by   : {confidence['failed_checks']}")
    else:
        print(f"  AI skip reason    : {confidence['skip_reason']}")
    print(f"  {'─'*50}")

print(f"\nAll tables processed.")
print(f"Total checks   : {len(all_result_records)}")
print(f"Total runs     : {len(all_run_records)}")

## Step 6: Write to Output Tables

In [0]:
print("Writing reconciliation results...")

for run_record in all_run_records:
    write_reconciliation_run(run_record)

write_reconciliation_results(all_result_records)

print("\nWrite complete.")

## Step 7: View Results

In [0]:
print("=" * 60)
print(f"RECONCILIATION RESULTS — {BUSINESS_DATE}")
print("=" * 60)

# Run summary
print("\n1. Reconciliation Run Summary:\n")
run_ids = [r["reconciliation_run_id"] for r in all_run_records]
run_filter = "','".join(run_ids)

spark.sql(f"""
    SELECT
        reconciliation_run_id,
        REGEXP_EXTRACT(
            source_fully_qualified_name,
            '[^.]+$', 0
        )                                              AS source_table,
        business_date,
        overall_status,
        total_checks_run,
        total_checks_passed,
        total_checks_warned,
        total_checks_critical,
        ai_enrichment_status,
        agent1_validation_run_id IS NOT NULL          AS has_agent1_context,
        ROUND(reconciliation_duration_seconds, 2)      AS duration_sec
    FROM {RECONCILIATION_RUN}
    WHERE reconciliation_run_id IN ('{run_filter}')
    ORDER BY source_fully_qualified_name
""").show(truncate=False)

# Detailed check results
print("\n2. Individual Check Results:\n")
spark.sql(f"""
    SELECT
        REGEXP_EXTRACT(
            source_fully_qualified_name,
            '[^.]+$', 0
        )                                              AS source_table,
        from_layer,
        to_layer,
        check_type,
        metric_name,
        check_status,
        ROUND(source_value, 2)                         AS source_value,
        ROUND(target_value, 2)                         AS target_value,
        ROUND(expected_value, 2)                       AS expected_value,
        ROUND(deviation_percentage, 4)                 AS deviation_pct,
        agent1_overall_status,
        threshold_warning,
        threshold_critical
    FROM {RECONCILIATION_RESULTS}
    WHERE reconciliation_run_id IN ('{run_filter}')
    ORDER BY
        source_fully_qualified_name,
        CASE check_status
            WHEN 'CRITICAL' THEN 1
            WHEN 'WARNING'  THEN 2
            WHEN 'PASS'     THEN 3
        END,
        check_type
""").show(50, truncate=False)

# Failed checks only
print("\n3. Failed Checks Only:\n")
spark.sql(f"""
    SELECT
        REGEXP_EXTRACT(
            source_fully_qualified_name,
            '[^.]+$', 0
        )                                              AS source_table,
        from_layer,
        to_layer,
        check_type,
        metric_name,
        check_status,
        ROUND(source_value, 2)                         AS source_value,
        ROUND(target_value, 2)                         AS target_value,
        ROUND(deviation_percentage, 4)                 AS deviation_pct,
        agent1_overall_status,
        check_detail
    FROM {RECONCILIATION_RESULTS}
    WHERE reconciliation_run_id IN ('{run_filter}')
      AND check_status IN ('WARNING', 'CRITICAL')
    ORDER BY
        CASE check_status WHEN 'CRITICAL' THEN 1 ELSE 2 END,
        check_type
""").show(50, truncate=False)

## Final Summary

In [0]:
print("\n" + "=" * 60)
print("AGENT 2 RECONCILIATION COMPLETE")
print("=" * 60)
print(f"Business date     : {BUSINESS_DATE}")
print(f"Tables reconciled : {len(all_run_records)}")
print(f"Total checks run  : {len(all_result_records)}")
print(f"Completed at      : {datetime.now()}")
print()

for run in all_run_records:
    print(f"  {run['source_fully_qualified_name'].split('.')[-1]}")
    print(f"    Status   : {run['overall_status']}")
    print(f"    Passed   : {run['total_checks_passed']}")
    print(f"    Warning  : {run['total_checks_warned']}")
    print(f"    Critical : {run['total_checks_critical']}")
    print(f"    AI       : {run['ai_enrichment_status']}")
    print()

print("=" * 60)